# LLM Pipeline for View Prediction Analysis

This notebook uses Gemini 3.1 Flash lite to analyze video performance based on the dataset created by the `numeric_inference` notebook. It generates qualitative descriptions of video performance drivers and PCA dimensions to assist in view prediction.

## Setup and Dependencies

In [ ]:
# Install necessary libraries
!pip install -q -U google-generativeai

import os
import json
import time
import numpy as np
import pandas as pd
from google.colab import drive, userdata
import google.generativeai as genai
from datetime import datetime

# Mount Google Drive
drive.mount('/content/drive')

# Configure Gemini
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-3.1-flash-lite-latest'

# Constants
BASE_PATH = '/content/drive/MyDrive/Graphiko/exports/base_data/latest/'
INPUT_FILE = os.path.join(BASE_PATH, 'top_significant_channels_eval.json')
OUTPUT_FILE = os.path.join(BASE_PATH, 'llm_analysis_results.json')
CACHE_FILE = os.path.join(BASE_PATH, 'gemini_cache.json')

print(f"Using input file: {INPUT_FILE}")

## Utility Functions

Includes caching, retry logic, and request rate limiting.

In [ ]:
def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_cache(cache):
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=4)

def get_gemini_completion(prompt, cache, key, retries=3, sleep_time=2):
    if key in cache:
        return cache[key]
    
    model = genai.GenerativeModel(MODEL_NAME)
    
    for i in range(retries):
        try:
            response = model.generate_content(prompt)
            text = response.text
            cache[key] = text
            save_cache(cache)
            time.sleep(sleep_time)
            return text
        except Exception as e:
            print(f"Error on attempt {i+1}: {e}")
            if "context" in str(e).lower():
                print("Context limit likely exceeded. Attempting to reduce prompt size...")
                lines = prompt.split('\n')
                if len(lines) > 20:
                    prompt = '\n'.join(lines[:10] + lines[-10:])
            if i < retries - 1:
                time.sleep(sleep_time * (i + 1))
            else:
                print("Max retries reached. Stopping pipeline.")
                raise e
    return None

## 1. Global Performance Analysis

Analyzing top 25 and bottom 25 videos per channel to identify success drivers. These descriptions will be used to predict views on the testing data.

In [ ]:
with open(INPUT_FILE, 'r') as f:
    dataset = json.load(f)

cache = load_cache()
global_explanations = {}

for channel in dataset:
    channel_name = channel['channel_name']
    videos = sorted(channel['train_videos'], key=lambda x: x['actual_views'], reverse=True)
    
    top_25 = [v['title'] for v in videos[:25]]
    bottom_25 = [v['title'] for v in videos[-25:]]
    
    prompt = f"""Analyze the following YouTube video titles from the channel '{channel_name}'.

Top performing videos (most views):
""" + "\n".join(top_25) + """

Bottom performing videos (least views):
""" + "\n".join(bottom_25) + """

Task: Based on these examples, explain what makes videos perform well or poorly on this channel. Provide a concise, predictive description that can be used to estimate views for new titles.
"""
    
    print(f"\n--- Analyzing Performance for {channel_name} ---")
    explanation = get_gemini_completion(prompt, cache, f"perf_{channel['channel_id']}")
    global_explanations[channel['channel_id']] = explanation
    print(f"Result Summary: {explanation[:200]}...")

## 2. PCA Dimension Descriptions

Generating descriptions for each of the 15 PCA dimensions using the MECE framework. We contrast the top 50 most positive and top 50 most negative videos for each dimension.

In [ ]:
all_train_videos = []
for channel in dataset:
    all_train_videos.extend(channel['train_videos'])

num_dims = 15
dimension_descriptions = []

for d in range(num_dims):
    sorted_by_dim = sorted(all_train_videos, key=lambda x: x['reduced_embedding'][d])
    
    bottom_50 = []
    seen_b = set()
    for v in sorted_by_dim:
        if v['title'] not in seen_b:
            bottom_50.append(v['title'])
            seen_b.add(v['title'])
        if len(bottom_50) == 50: break

    top_50 = []
    seen_t = set()
    for v in reversed(sorted_by_dim):
        if v['title'] not in seen_t:
            top_50.append(v['title'])
            seen_t.add(v['title'])
        if len(top_50) == 50: break
    
    past_context = "\n".join([f"Dimension {i}: {desc[:150]}..." for i, desc in enumerate(dimension_descriptions)])
    
    prompt = f"""You are defining latent semantic dimensions derived from PCA on YouTube titles. This is Dimension {d}.

Constraint: PCA dimensions are orthogonal. Use the MECE framework to ensure your description of Dimension {d} does not overlap with previous dimensions.

Previous Dimension Context:
{past_context if past_context else 'None'}

Representative titles for Dimension {d}:
--- HIGHEST POSITIVE EMBEDDING ---
""" + "\n".join(top_50) + """

--- HIGHEST NEGATIVE EMBEDDING ---
""" + "\n".join(bottom_50) + """

Task: Describe what semantic concept Dimension {d} captures. Contrast what the positive end represents vs. the negative end. Keep it distinct from the previous dimensions.
"""
    
    print(f"\n--- Describing Dimension {d} ---")
    desc = get_gemini_completion(prompt, cache, f"dim_{d}")
    dimension_descriptions.append(desc)
    print(f"Result Summary: {desc[:200]}...")

## 3. Channel-Specific Driver Explanation (Top 5 Dimensions)

Generating a detailed explanation of what drives performance for each channel using its top 5 most significant dimensions.

In [ ]:
channel_dim_analysis = {}

for channel in dataset:
    p_values = np.array(channel['model']['p_values'][1:])
    coeffs = channel['model']['coefficients']
    significant_indices = np.argsort(p_values)[:5]
    
    dim_info = []
    for idx in significant_indices:
        impact = "POSITIVE" if coeffs[idx] > 0 else "NEGATIVE"
        mag = abs(coeffs[idx])
        dim_info.append(f"Dimension {idx} (P-value: {p_values[idx]:.4e}, Effect: {impact}, Magnitude: {mag:.4f})")
        dim_info.append(f"Description: {dimension_descriptions[idx]}")
        sorted_vids = sorted(channel['train_videos'], key=lambda x: x['reduced_embedding'][idx])
        dim_info.append(f"- Top channel titles in this dim: " + ", ".join([v['title'] for v in sorted_vids[-10:]]))
        dim_info.append(f"- Bottom channel titles in this dim: " + ", ".join([v['title'] for v in sorted_vids[:10]]))
        dim_info.append("")

    prompt = f"""Generate an explanation of what drives performance for the YouTube channel '{channel['channel_name']}'.

We have identified the 5 most statistically significant PCA dimensions for this channel's view count:

""" + "\n".join(dim_info) + f"""

Task: Using the information above, explain what drives performance for '{channel['channel_name']}'. 
Requirements:
1. Order dimensions from most significant to least.
2. Clearly explain if each dimension affects performance positively or negatively and to what extent.
3. Use the provided examples to ground the explanation.
4. Produce a single cohesive description that will be used for view prediction.
"""
    
    print(f"\n--- Generating Channel Driver Analysis: {channel['channel_name']} ---")
    analysis = get_gemini_completion(prompt, cache, f"chan_dim_{channel['channel_id']}")
    channel_dim_analysis[channel['channel_id']] = analysis
    print(f"Result Summary: {analysis[:200]}...")

## Export Results

In [ ]:
final_output = {
    "global_performance_descriptions": global_explanations,
    "dimension_definitions": dimension_descriptions,
    "channel_significant_dimension_analysis": channel_dim_analysis,
    "metadata": {
        "generated_at": datetime.now().isoformat(),
        "model": MODEL_NAME,
        "input_file": INPUT_FILE,
        "num_dimensions": num_dims
    }
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(final_output, f, indent=4)

print(f"\n[SUCCESS] Analysis results exported to {OUTPUT_FILE}")

## Stats and Summary

In [ ]:
print(f"Summary of processing:")
print(f"- Channels Analyzed: {len(global_explanations)}")
print(f"- Dimensions Described: {len(dimension_descriptions)}")
print(f"- Timestamp: {final_output['metadata']['generated_at']}")
print(f"- Output Path: {OUTPUT_FILE}")